# Explore: predict *play* instead of *own*

**Question:** if positives are games a user actually played (≥ X minutes), how do simple
baselines behave, and which threshold should task A use?

This notebook drove the switch from "own" to "play". `prep` now builds the 120-minute
task itself (`play_*` in the cache); here we rebuild the same task **for any threshold**
with exactly prep's rules, so thresholds can be compared side by side:

- **positives:** held-out *played* games (leave-one-out among played games, `[split.play]`)
- **negatives:** 99 popularity-weighted games the user owns in **no** split
- **owned but under the threshold:** stays in train as context (for users who have
  played games); never a positive or a negative

Threshold `0` is the old ownership task (any owned game is a positive), for comparison.

In [14]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace

import numpy as np
import pandas as pd

from assignment_2.modules.config import get_config
from assignment_2.modules.core import (
    SimMetric,
    interaction_matrix,
    item_similarity,
    ranking_per_user,
    user_similarity,
)
from assignment_2.modules.data import load_prepped, ranking_candidates
from assignment_2.modules.data.split import split_by_user

get_config.cache_clear()
cfg = get_config()
t = load_prepped(cfg)
owned = t["owned"]  # every owned (user, game), unsplit
print(f"sample={cfg.sample.tag}  negatives={cfg.evaluation.negatives}  "
      f"{len(owned):,} ownership rows, {owned.user_id.nunique():,} users")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
sample=users_0.1  negatives=popularity  490,861 ownership rows, 7,058 users


## 1. Build a "predict play" task for a threshold

Validation only: we're exploring, so test stays untouched. `play_task` follows
`prep.py` step for step, so at `cfg.split.play.min_minutes` it must reproduce the cached
`play_valid_candidates` exactly (checked below).

In [15]:
THRESHOLDS = [0, 5, 30, 60, 120, 240]   # minutes; 0 = any owned game counts (old task)
K = 10


def play_task(min_minutes: int) -> dict:
    """Train rows, held-out played games, and validation candidates for one threshold."""
    is_played = owned.playtime_forever >= min_minutes   # min_minutes = 0: every owned game
    played, unplayed = owned[is_played], owned[~is_played]

    s = split_by_user(played, cfg.split.play, cfg.seed)  # leave-one-out among played games
    # as in prep: under-threshold rows join train only for users who have played games
    extra = unplayed[unplayed.user_id.isin(s.train.user_id)]
    s = replace(s, train=pd.concat([s.train, extra], ignore_index=True))

    cand = ranking_candidates(
        held_out=s.valid,
        train=s.train,
        owned=owned,                          # never sample anything the user owns
        n_negatives=cfg.evaluation.n_negatives,
        seed=cfg.seed + 1,                    # prep's seed for the valid candidates
        negatives=cfg.evaluation.negatives,
    )
    return {"train": s.train, "valid": s.valid, "cand": cand, "min_minutes": min_minutes,
            "played_share": is_played.mean()}


tasks = {m: play_task(m) for m in THRESHOLDS}

**Consistency check:** at prep's threshold, the rebuilt task must be the cached one, so
the numbers below are the ones every model will be scored on.

In [16]:
m0 = cfg.split.play.min_minutes
keys = ["user_id", "item_id", "label"]
rebuilt = tasks[m0]["cand"][keys].sort_values(keys).reset_index(drop=True)
cached = t["play_valid_candidates"][keys].sort_values(keys).reset_index(drop=True)
assert rebuilt.equals(cached), "play_task no longer matches prep.py"
assert len(tasks[m0]["train"]) == len(t["play_train"])
print(f"{m0} min: rebuilt task matches the cache ({len(cached):,} candidate rows)")

120 min: rebuilt task matches the cache (594,600 candidate rows)


## 2. What each threshold costs

A user needs ≥ 3 *played* games to have one held out, so higher thresholds evaluate
fewer users.

In [17]:
n_eval_0 = tasks[0]["cand"].user_id.nunique()
cost = pd.DataFrame(
    {
        m: {
            "owned rows counted as played": f"{d['played_share']:.1%}",
            "train rows": f"{len(d['train']):,}",
            "users evaluated": f"{d['cand'].user_id.nunique():,}",
            "share of users at 0 min": f"{d['cand'].user_id.nunique() / n_eval_0:.0%}",
        }
        for m, d in tasks.items()
    }
).T.rename_axis("min minutes")
cost

,owned rows counted as played,train rows,users evaluated,share of users at 0 min
min minutes,,,,
0,100.0%,"477,777","6,535",100%
5,60.3%,"477,899","6,226",95%
30,51.4%,"477,954","6,136",94%
60,45.5%,"477,894","6,052",93%
120,38.4%,"477,901","5,946",91%
240,30.3%,"477,850","5,775",88%


## 3. Baselines

Each baseline is a function `scores(task) -> np.ndarray`, one score per candidate row, in
order. `evaluate` turns scores into Hit@10 / NDCG@10 / MRR averaged over users.

In [18]:
def evaluate(task: dict, scores: np.ndarray) -> pd.Series:
    c = task["cand"]
    per_user = ranking_per_user(c.user_id, c.label, scores, k=K, seed=cfg.seed)
    return pd.Series({m: v.mean() for m, v in per_user.items()})


def random_scores(task: dict) -> np.ndarray:
    return np.random.default_rng(cfg.seed).random(len(task["cand"]))


def owner_popularity(task: dict) -> np.ndarray:
    """How many train users own each candidate (played or not)."""
    counts = task["train"].item_id.value_counts()
    return task["cand"].item_id.map(counts).fillna(0).to_numpy(float)

### 3a. Play popularity
**Idea:** rank by how many train users *played* each game (≥ the threshold), not just
own it. If positives are played games, does counting plays beat counting owners?

In [19]:
def play_popularity(task: dict) -> np.ndarray:
    """How many train users *played* each candidate (at this task's threshold)."""
    train = task["train"]
    played = train[train.playtime_forever >= task["min_minutes"]]
    counts = played.item_id.value_counts()
    return task["cand"].item_id.map(counts).fillna(0).to_numpy(float)

### 3b. Play rate
**Idea:** a game owned by 1,000 people but played by 100 is less promising than one owned
by 200 and played by 180. Score = share of a game's owners who played it, shrunk toward
the overall rate, `(plays + a * overall_rate) / (owners + a)` with `a = 10`, so a game
with one owner who played it doesn't score 100%.

In [20]:
def play_rate(task: dict, a: float = 10.0) -> np.ndarray:
    """Share of a game's train owners who played it, shrunk toward the overall rate."""
    train = task["train"]
    owners = train.item_id.value_counts()                     # everyone who owns it
    plays = (train[train.playtime_forever >= task["min_minutes"]]
             .item_id.value_counts()
             .reindex(owners.index, fill_value=0))            # same games, 0 if never played
    overall = plays.sum() / owners.sum()                      # the rate across all ownership

    rate = (plays + a * overall) / (owners + a)               # shrunk toward `overall`
    return task["cand"].item_id.map(rate).fillna(overall).to_numpy(float)

### 3c. Personal: similarity to what the user played
**Idea:** score a candidate by its max Jaccard similarity to the games *this user*
played in train, where a game is represented by the set of users who played it.
This is the first **personalized** baseline: everything above ranks the same for every user.

In [21]:
def item_knn(task: dict, metric: SimMetric = "jaccard") -> np.ndarray:
    """max similarity between each candidate and the games the user played (item-based)."""
    train = task["train"]
    played = train[train.playtime_forever >= task["min_minutes"]]
    X, users, items = interaction_matrix(played)
    S = item_similarity(X, metric)

    cand = task["cand"]
    u_idx = users.get_indexer(cand.user_id)   # -1 = user with no played games
    i_idx = items.get_indexer(cand.item_id)   # -1 = candidate nobody played

    scores = np.zeros(len(cand))
    for u, rows in pd.Series(np.arange(len(cand))).groupby(u_idx).groups.items():
        rows = np.asarray(rows)
        history = X[u].indices                           # games this user played
        ok = rows[i_idx[rows] >= 0] if u >= 0 else rows[:0]
        if len(ok) == 0 or len(history) == 0:
            continue                                     # nothing to compare: score stays 0
        sims = S[i_idx[ok]][:, history]                  # candidates × history
        scores[ok] = sims.max(axis=1).toarray().ravel()
    return scores

In [22]:
def user_knn(task: dict, metric: SimMetric = "jaccard") -> np.ndarray:
    """Sum of the user's similarity to every user who played the candidate (user-based)."""
    train = task["train"]
    played = train[train.playtime_forever >= task["min_minutes"]]
    X, users, items = interaction_matrix(played)
    S = user_similarity(X, metric)
    R = (S @ X).tocsr()            # R[u, i] = Σ over users v who played i of sim(u, v)

    cand = task["cand"]
    u_idx = users.get_indexer(cand.user_id)
    i_idx = items.get_indexer(cand.item_id)
    ok = (u_idx >= 0) & (i_idx >= 0)
    scores = np.zeros(len(cand))
    scores[ok] = np.asarray(R[u_idx[ok], i_idx[ok]]).ravel()
    return scores

## 4. Compare across thresholds

Jaccard is the slow one (a few minutes per threshold); set `SLOW = False` to skip it.

In [23]:
BASELINES = {
    "random": random_scores,
    "owner popularity": owner_popularity,
    "play popularity": play_popularity,
    "play rate": play_rate,
    "item knn (jaccard)": item_knn,
    "item knn (cosine)": lambda t: item_knn(t, "cosine"),
    "user knn (cosine)": lambda t: user_knn(t, "cosine"),
}

results = pd.concat(
    {
        (m, name): evaluate(task, fn(task))
        for m, task in tasks.items()
        for name, fn in BASELINES.items()
    },
    names=["min_minutes", "baseline"],
).unstack().round(3)
results

hit@10  ndcg@10    mrr
min_minutes baseline                                  
0           item knn (cosine)    0.500    0.368  0.345
            item knn (jaccard)   0.502    0.366  0.342
            owner popularity     0.293    0.169  0.153
            play popularity      0.293    0.169  0.153
            play rate            0.100    0.046  0.052
            random               0.091    0.041  0.049
            user knn (cosine)    0.335    0.198  0.178
5           item knn (cosine)    0.420    0.256  0.228
            item knn (jaccard)   0.426    0.257  0.227
            owner popularity     0.296    0.166  0.150
            play popularity      0.324    0.188  0.169
            play rate            0.245    0.127  0.117
            random               0.090    0.041  0.049
            user knn (cosine)    0.360    0.208  0.184
30          item knn (cosine)    0.453    0.276  0.243
            item knn (jaccard)   0.457    0.275  0.242
            owner popularity     0.328    0.189  0.170
            play popularity      0.363    0.214  0.193
            play rate            0.301    0.160  0.142
            random               0.089    0.041  0.049
            user knn (cosine)    0.401    0.234  0.206
60          item knn (cosine)    0.460    0.287  0.254
            item knn (jaccard)   0.466    0.287  0.253
            owner popularity     0.339    0.198  0.177
            play popularity      0.371    0.224  0.203
            play rate            0.319    0.175  0.157
            random               0.090    0.040  0.049
            user knn (cosine)    0.409    0.246  0.219
120         item knn (cosine)    0.493    0.304  0.266
            item knn (jaccard)   0.497    0.303  0.264
            owner popularity     0.357    0.213  0.190
            play popularity      0.405    0.246  0.220
            play rate            0.350    0.192  0.169
            random               0.090    0.041  0.049
            user knn (cosine)    0.440    0.268  0.237
240         item knn (cosine)    0.512    0.314  0.274
            item knn (jaccard)   0.518    0.315  0.274
            owner popularity     0.362    0.217  0.195
            play popularity      0.425    0.255  0.227
            play rate            0.370    0.198  0.172
            random               0.090    0.041  0.049
            user knn (cosine)    0.465    0.277  0.242

### Task A baseline table (valid, cached candidates)
The rows at prep's threshold are the baseline rows for the results table and `plan.md` §4.

In [24]:
results.loc[m0]

,hit@10,ndcg@10,mrr
baseline,,,
item knn (cosine),0.493,0.304,0.266
item knn (jaccard),0.497,0.303,0.264
owner popularity,0.357,0.213,0.190
play popularity,0.405,0.246,0.220
play rate,0.350,0.192,0.169
random,0.090,0.041,0.049
user knn (cosine),0.440,0.268,0.237


## 5. Questions to answer
- Does **owner popularity** change as the threshold rises? (If played games are a
  different set from popular-to-own games, it should.)
- Do **play popularity** or **play rate** beat owner popularity above 0 minutes? If so,
  playtime carries signal the ownership task couldn't use.
- How much does **item kNN** (personal) add over the best popularity-style score, and at
  which threshold is that gap largest? That's where personalized models have the most room.
- Which similarity works best (**Jaccard vs cosine**), and does **user kNN** beat item kNN?
- How many users does each threshold cost (§2)? Is 120 minutes still a large enough sample,
  and does anything between 30 and 240 minutes look clearly better?

**Takeaways:**
- At 0 min (the old ownership task), play popularity = owner popularity, and item kNN's
  +0.21 margin is largely bundle recovery.
- Above 0 min, the personalization margin (item kNN − play popularity) is flat at
  ~+0.09–0.10 from 5 to 240 min, so the data doesn't pick a threshold; 120 min is chosen
  for meaning (Steam's refund window) and keeps 91% of users (5,946 of 6,535).
- Scores rise with the threshold because longer-played games skew popular. Compare
  margins within a threshold, not scores across thresholds.
- Play popularity's lead over owner popularity grows with the threshold (+0.028 at 5 min,
  +0.048 at 120, +0.063 at 240): who *played* a game beats who owns it.
- The sparse item kNN reproduces the original pure-Python Jaccard baseline exactly at
  every threshold.
- Item kNN: Jaccard and cosine are effectively tied (Hit@10 0.497 vs 0.493 at 120 min;
  cosine slightly ahead on NDCG/MRR). Needs paired bootstrap intervals to separate.
- User kNN (cosine, summed over similar users) beats play popularity (0.440 vs 0.405) but
  trails item kNN by ~0.055 above 0 min, and by 0.17 at 0 min, where item–item bundle
  structure dominates. The sum also rewards widely played games (a popularity bias);
  top-k neighbors or normalization would test that.
- Task A baselines (valid, cached candidates, 120 min, Hit@10): random 0.090, play rate
  0.350, owner popularity 0.357, play popularity 0.405, user kNN 0.440,
  **item kNN (Jaccard) 0.497**.